In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
# --- 0. THIẾT LẬP CHUNG ---
CUTOFF_DAY = 135
RANDOM_SEED = 42

DATAPATH = "../../data/"
df_student_info = pd.read_csv(DATAPATH + 'studentInfo.csv')
df_assessments = pd.read_csv(DATAPATH + 'assessments.csv') 
df_student_assessment = pd.read_csv(DATAPATH + 'studentAssessment.csv')

## 4. Nhóm tính năng đánh giá & Học tập (Assessment Features)
Quy tắc xác định **Safe Assessment**: Chỉ tính các bài có `deadline <= CUTOFF_DAY`. Toàn bộ bài thi cuối kỳ (Exam) bị loại bỏ vì gây rò rỉ dữ liệu 100%.

* **weighted_score_before_cutoff**: Tính theo công thức `sum(score × weight) / sum(weight)`.
* **tma_submission_rate & cma_submission_rate**: 
   * Công thức: `n_submitted / n_safe_in_module`.
   * Nếu module không có loại bài đó trước ngày 135: Điền `-1`.
   * Nếu có bài nhưng sinh viên không nộp: Điền `0`.
* **avg_days_before_deadline**: Trung bình số ngày nộp trước hạn (Giữ nguyên giá trị âm nếu nộp trễ, không thực hiện clip).
* **n_missing_submission**: Tổng số bài đánh giá "Safe" mà sinh viên bỏ lỡ.
* **Xử lý Edge Cases**:
   * **Module GGG**: Do trọng số TMA = 0 nên không tính được điểm tích lũy. Thêm cột `has_weighted_score = 0` để mô hình phân biệt và sử dụng `TMA submission count` làm tín hiệu thay thế.
   * **Module AAA, EEE**: Điền `-1` cho `cma_submission_rate` do không có bài CMA trước ngày 135.

In [8]:
# MỤC 4: NHÓM TÍNH NĂNG ĐÁNH GIÁ & HỌC TẬP
# ==========================================
def engineer_assessment_features(df_info, df_assess, df_stud_assess):
    # 1. Lọc Safe Assessment
    safe_assess = df_assess[
        (df_assess['date'] <= CUTOFF_DAY) & 
        (df_assess['assessment_type'] != 'Exam')
    ].copy()
    
    # Kết hợp điểm của sinh viên với thông tin bài đánh giá
    merged_assess = pd.merge(df_stud_assess, safe_assess, on='id_assessment', how='inner')
    
    # 2. Tính số ngày nộp trước hạn: deadline (date) - date_submitted
    # Dương: nộp sớm, Âm: nộp trễ
    merged_assess['days_before_deadline'] = merged_assess['date'] - merged_assess['date_submitted']
    
    # Tính điểm trọng số: score * weight
    merged_assess['weighted_score_part'] = merged_assess['score'] * merged_assess['weight']
    
    # 3. Aggregate dữ liệu theo từng sinh viên và môn học
    assess_agg = merged_assess.groupby(['id_student', 'code_module', 'code_presentation']).agg(
        total_weighted_score_part=('weighted_score_part', 'sum'),
        total_weight=('weight', 'sum'),
        avg_days_before_deadline=('days_before_deadline', 'mean'),
        n_submitted=('id_assessment', 'count')
    ).reset_index()
    
    # Tính weighted_score_before_cutoff
    assess_agg['weighted_score_before_cutoff'] = np.where(
        assess_agg['total_weight'] > 0,
        assess_agg['total_weighted_score_part'] / assess_agg['total_weight'],
        0
    )
    
    # 4. Tính tỷ lệ nộp bài (TMA & CMA) và xử lý Edge Cases
    # Đếm số lượng bài Safe Assessment yêu cầu của mỗi module
    module_req = safe_assess.groupby(['code_module', 'code_presentation', 'assessment_type']).size().unstack(fill_value=0).reset_index()
    if 'CMA' not in module_req.columns: module_req['CMA'] = 0
    if 'TMA' not in module_req.columns: module_req['TMA'] = 0
    
    # Đếm số bài sinh viên đã nộp theo loại
    stud_submitted = merged_assess.groupby(['id_student', 'code_module', 'code_presentation', 'assessment_type']).size().unstack(fill_value=0).reset_index()
    if 'CMA' not in stud_submitted.columns: stud_submitted['CMA'] = 0
    if 'TMA' not in stud_submitted.columns: stud_submitted['TMA'] = 0
    
    # Merge yêu cầu và thực tế
    rates_df = pd.merge(stud_submitted, module_req, on=['code_module', 'code_presentation'], suffixes=('_sub', '_req'))
    
    # Logic tỷ lệ: Nếu req == 0 -> -1; Nếu req > 0 -> sub / req
    rates_df['tma_submission_rate'] = np.where(rates_df['TMA_req'] == 0, -1, rates_df['TMA_sub'] / rates_df['TMA_req'])
    rates_df['cma_submission_rate'] = np.where(rates_df['CMA_req'] == 0, -1, rates_df['CMA_sub'] / rates_df['CMA_req'])
    rates_df['n_missing_submission'] = (rates_df['TMA_req'] + rates_df['CMA_req']) - (rates_df['TMA_sub'] + rates_df['CMA_sub'])
    
    # 5. Xử lý Edge Case Module GGG
    assess_agg['has_weighted_score'] = np.where(assess_agg['code_module'] == 'GGG', 0, 1)
    
    # Gộp tất cả vào dataframe gốc
    df_final = pd.merge(df_info, assess_agg[['id_student', 'code_module', 'code_presentation', 'weighted_score_before_cutoff', 'avg_days_before_deadline', 'has_weighted_score']], on=['id_student', 'code_module', 'code_presentation'], how='left')
    df_final = pd.merge(df_final, rates_df[['id_student', 'code_module', 'code_presentation', 'tma_submission_rate', 'cma_submission_rate', 'n_missing_submission']], on=['id_student', 'code_module', 'code_presentation'], how='left')
    
    # Điền khuyết cho những sinh viên không có hoạt động đánh giá nào
    df_final.fillna({
        'weighted_score_before_cutoff': 0,
        'avg_days_before_deadline': 0,
        'has_weighted_score': 1,
        'tma_submission_rate': 0, 
        'cma_submission_rate': 0,
        'n_missing_submission': 0 # Sẽ cần điều chỉnh tùy thuộc vào số module thực tế nếu sinh viên trống hoàn toàn
    }, inplace=True)
    
    # Áp dụng lại quy tắc -1 cho AAA, EEE (CMA) và GGG nếu bị fill nhầm thành 0 do fillna
    df_final.loc[df_final['code_module'].isin(['AAA', 'EEE']), 'cma_submission_rate'] = -1
    df_final.loc[df_final['code_module'] == 'GGG', 'has_weighted_score'] = 0

    return df_final

In [ ]:
df_master = engineer_assessment_features(df_student_info, df_assessments, df_student_assessment)

## 5. Nhóm tính năng bối cảnh & Nhân khẩu học
* **module_semester**: Kết hợp mã môn và học kỳ (Ví dụ: AAA_B) -> Sử dụng Label Encoding.
* **num_of_prev_attempts**: Giữ nguyên do có tương quan mạnh với rủi ro trượt/rút môn.
* **Mã hóa thứ tự (Ordinal Encoding)**:
   * `highest_education`: Sắp xếp theo trình độ từ thấp đến cao.
   * `age_band`: 0-35 < 35-55 < 55+.
   * `imd_band`: 0-10% < ... < 90-100% < Unknown.
* **Loại bỏ**: Xóa tính năng `gender` do không đóng góp giá trị dự báo.

In [ ]:

def engineer_demographic_features(df):
    # Tạo module_semester
    df['module_semester'] = df['code_module'] + '_' + df['code_presentation']
    
    # Ordinal Encoding Dictionaries
    edu_map = {
        'No Formal quals': 0, 
        'Lower Than A Level': 1, 
        'A Level or Equivalent': 2, 
        'HE Qualification': 3, 
        'Post Graduate Qualification': 4
    }
    age_map = {'0-35': 0, '35-55': 1, '55<=': 2}
    imd_map = {
        '0-10%': 0, '10-20': 1, '20-30%': 2, '30-40%': 3, '40-50%': 4,
        '50-60%': 5, '60-70%': 6, '70-80%': 7, '80-90%': 8, '90-100%': 9,
        'Unknown': 10 # Như đã quy định ở Mục 1
    }
    
    df['highest_education'] = df['highest_education'].map(edu_map)
    df['age_band'] = df['age_band'].map(age_map)
    df['imd_band'] = df['imd_band'].map(imd_map)
    
    # Loại bỏ gender
    if 'gender' in df.columns:
        df = df.drop(columns=['gender'])
        
    return df

In [ ]:
df_master = engineer_demographic_features(df_master)

## 6. Danh mục kiểm tra & Tiền xử lý (Modeling Prep)
* **Stratified Split**: Chia tập dữ liệu dựa trên nhãn mục tiêu và mã môn học để đảm bảo tính đồng nhất.
* **Student-level Split**: Đảm bảo tất cả bản ghi của cùng một `id_student` phải nằm hoàn toàn trong tập Train hoặc tập Test để tránh rò rỉ thông tin sinh viên.
* **Scaling**:
    * Với mô hình cây (Random Forest, XGBoost): Không cần scaling.
    * Với mô hình tuyến tính: Sử dụng `StandardScaler`, chỉ **fit trên tập Train** và áp dụng (transform) cho tập Test.

In [ ]:
# MỤC 6 & 7: CHIA TẬP VÀ TIỀN XỬ LÝ (PIPELINE)
# ==========================================
def split_and_scale_data(df):
    # Mã hóa module_semester (Label Encoding)
    le = LabelEncoder()
    df['module_semester_encoded'] = le.fit_transform(df['module_semester'])
    
    # 1. Student-level Split kết hợp Stratified
    # Để tránh rò rỉ, chúng ta lấy danh sách id_student duy nhất
    unique_students = df[['id_student', 'final_result', 'code_module']].drop_duplicates(subset=['id_student'])
    
    # Tạo cột stratification = final_result + code_module
    unique_students['stratify_col'] = unique_students['final_result'] + '_' + unique_students['code_module']
    
    # Xử lý các nhóm stratify quá nhỏ (chỉ có 1 mẫu) để tránh lỗi train_test_split
    val_counts = unique_students['stratify_col'].value_counts()
    valid_stratify = val_counts[val_counts > 1].index
    unique_students['stratify_col'] = np.where(unique_students['stratify_col'].isin(valid_stratify), unique_students['stratify_col'], 'Other')

    # Chia tập trên danh sách sinh viên
    train_studs, test_studs = train_test_split(
        unique_students['id_student'], 
        test_size=0.2, 
        random_state=RANDOM_SEED, 
        stratify=unique_students['stratify_col']
    )
    
    # Tách dataframe chính dựa trên id_student đã chia
    df_train = df[df['id_student'].isin(train_studs)].copy()
    df_test = df[df['id_student'].isin(test_studs)].copy()
    
    # 2. Scaling (Chỉ áp dụng fit trên Train)
    # Liệt kê các cột dạng số liên tục cần scale (loại bỏ id, biến phân loại, biến ordinal)
    cols_to_scale = [
        'weighted_score_before_cutoff', 'avg_days_before_deadline', 
        'tma_submission_rate', 'cma_submission_rate', 'n_missing_submission',
        'num_of_prev_attempts'
        # Thêm các cột từ Section 2 & 3 (clicks, active_days...) vào list này
    ]
    
    # Đảm bảo chỉ scale những cột tồn tại
    cols_to_scale = [c for c in cols_to_scale if c in df_train.columns]
    
    scaler = StandardScaler()
    
    # Fit & Transform trên Train
    df_train[cols_to_scale] = scaler.fit_transform(df_train[cols_to_scale])
    
    # Chỉ Transform trên Test
    df_test[cols_to_scale] = scaler.transform(df_test[cols_to_scale])
    
    return df_train, df_test, scaler

# ==========================================
# CÁCH CHẠY PIPELINE (MỤC 7)
# ==========================================
# 1 & 2 & 3 (Đã tính toán Sections 1-3 ở ngoài)
# Giả sử bạn đang có dataframe tổng df_master sau khi xong mục 3

In [ ]:
# train_set, test_set, fitted_scaler = split_and_scale_data(df_master)

## 7. Pipeline thực thi (Workflow Order)
Để tránh rò rỉ dữ liệu, thứ tự thực hiện phải được tuân thủ nghiêm ngặt:
1. Load dữ liệu và xử lý Semantic Nulls.
2. Lọc bỏ Exam và các dữ liệu phát sinh sau ngày 135.
3. Tính toán toàn bộ các tính năng (Sections 2-5).
4. Thực hiện Split dữ liệu (Train/Test).
   → Split theo Student-level (id_student) + Stratified (final_result × code_module)
5. Thực hiện Encoding và Scaling (Chỉ fit trên tập Train).
6. Huấn luyện và đánh giá mô hình.